# Zucchini Final Pipeline Benchmark — 3 Candidates, One Protocol

## Why the previous version of this notebook scored lower than your originals

Your `zucchini_ultimate_pipeline` reported **blend OOF R² = 0.9161** (HSV + CORAL). The first
draft of this comparison notebook scored below that. The cause was **not** the model — it was
five preprocessing/budget regressions introduced here that did not exist in your originals:

| # | Regression in the old draft | What your original did | Why it costs R² |
|---|---|---|---|
| 1 | **Geometric augmentation applied on the *encoded* image** (rotate/zoom/flip on HSV/LAB arrays) | Augmented the **RGB** image, *then* converted to the target encoding | HSV **Hue is circular** (179 wraps to 0). Interpolating it during rotate/zoom invents bogus hues across the wrap point. This poisons the HSV pipeline specifically — i.e. exactly the 0.9161 winner. **Single biggest cause.** |
| 2 | **Blind center-square crop** | No crop at all | The produce does not sit in a square region; a blind square crop discards produce pixels and keeps background. |
| 3 | **Bilateral filter** (`sigmaColor=35`) | No denoising | Bilateral smoothing flattens subtle chroma gradients — which *are* the scoring signal for a colour-grading task. |
| 4 | **Percentile 1–99 contrast stretch** | Plain min–max stretch | Clipping the top/bottom 1% removes the extreme colour values that separate adjacent score classes. |
| 5 | **Weaker budget**: ×13 augmentation, 220 epochs, ES patience 28 | ×16 augmentation, 300 epochs, ES patience 30 | Less regularisation and earlier stopping. |

**All five are fixed below**, and §2 runs an **empirical preprocessing ablation** so you can see
the R² cost of each choice on your own data rather than taking the table above on trust.

## What this notebook benchmarks

Three candidate pipelines, trained under **one identical protocol**:

| Pipeline | Backbone | Encoding | Head / Loss | Origin |
|---|---|---|---|---|
| **A** | 6-conv-block CNN | HSV | dual (ordinal+reg), composite CORAL | winner of `zucchini_ultimate_pipeline` |
| **B** | 4-conv-block CNN | LAB | dual (ordinal+reg), composite CORAL | winner of `Zucchini_CNN_Ultimate_Final` |
| **C** | `RegressionCNN` (6 stages, 16→512) | RGB | single regression head, Huber + rank | your `260305_cnn_regression_analysis` baseline |

**Identical for all three** (so the comparison is honest): content-aware cropping → colour-safe
filtering → the *same* stratified 5-fold splits → ×16 offline augmentation →
**300 epochs, ES patience 30, LR patience 8** → OOF prediction → same metric set, confusion
matrices, friezes and significance tests.


In [ ]:
import os, glob, time, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import cv2
import tifffile as tiff
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr, kendalltau, ttest_rel
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, cohen_kappa_score, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ---- Paths / runtime config ----
NOCC_ROOT = r'F:\Aymen\server\Database_CSVS\CSVS_Zucchini_202602_LaMojonena\Zucchini_COLOR_Evaluation\Zucchini_GREEN_FOR_SPAIN_on_LVS_platform\OUTPUT\cc\CSVS01'
OUT_DIR = 'compare_outputs_final3'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, 'checkpoints'), exist_ok=True)

QUICK_MODE = False
IMG_SIZE = (128, 128)
BATCH_SIZE = 16
K_FOLDS = 5

# Budget matched to the originals that produced R2=0.9161 (old draft used 220/28/8 -> part of the gap)
EPOCHS      = 40 if QUICK_MODE else 300
PATIENCE_ES = 8  if QUICK_MODE else 30
PATIENCE_LR = 4  if QUICK_MODE else 8

LR_INIT = 5e-4
LR_MIN = 1e-6
LR_FACTOR = 0.5
GRAD_CLIP = 2.0
BLEND_W_ORD = 0.7
NUM_CLASSES = 9
NUM_ORD = 8
SCORES = list(range(1, 10))
IDX_TO_SCORE = np.arange(1, 10, dtype=np.float32)

# ---- Preprocessing config (crop + filter are ablated empirically in section 2) ----
# crop_mode   'none'    = what the originals did
#             'center'  = blind square crop (the regression from the old draft)
#             'content' = produce-aware bounding-box crop
# filter_mode 'none' | 'median3' (colour-safe) | 'bilateral' (chroma-flattening)
# stretch     'minmax' (original) | 'percentile' (clips colour extremes)
PREPROCESS = {
    'crop_mode':    'content',
    'filter_mode':  'median3',
    'stretch_mode': 'minmax',
    'crop_margin':  0.04,
}

LEGACY_PREPROCESS = {
    'crop_mode':    'none',
    'filter_mode':  'none',
    'stretch_mode': 'minmax',
    'crop_margin':  0.0,
}

print(f'Budget      : {EPOCHS} epochs | ES patience {PATIENCE_ES} | LR patience {PATIENCE_LR}')
print(f'Folds       : {K_FOLDS} (identical splits shared by every pipeline)')
print(f'Preprocess  : {PREPROCESS}')

In [ ]:
_CV2_FORWARD = {'LAB': cv2.COLOR_RGB2LAB, 'HSV': cv2.COLOR_RGB2HSV, 'YCrCb': cv2.COLOR_RGB2YCrCb, 'RGB': None}


def stretch_to_uint8(img, mode='minmax'):
    if img.dtype == np.uint8:
        return img
    arr = img.astype(np.float32)
    if mode == 'percentile':
        lo, hi = np.percentile(arr, 1), np.percentile(arr, 99)
    else:
        lo, hi = arr.min(), arr.max()
    arr = np.clip((arr - lo) / (hi - lo + 1e-8), 0, 1)
    return (arr * 255).astype(np.uint8)


def center_square_crop(img):
    h, w = img.shape[:2]
    side = min(h, w)
    y0, x0 = (h - side) // 2, (w - side) // 2
    return img[y0:y0 + side, x0:x0 + side]


def content_crop(img_rgb, margin=0.04):
    """Bounding box of the produce via saturation+Otsu, so background is dropped
    without cutting produce pixels the way a blind square crop does."""
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    sat = hsv[..., 1]
    _, mask = cv2.threshold(sat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return img_rgb
    x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
    if w < 16 or h < 16:
        return img_rgb

    H, W = img_rgb.shape[:2]
    if (w * h) < 0.02 * H * W:          # mask collapsed -> not trustworthy
        return img_rgb

    mx, my = int(w * margin), int(h * margin)
    x0, y0 = max(x - mx, 0), max(y - my, 0)
    x1, y1 = min(x + w + mx, W), min(y + h + my, H)
    return img_rgb[y0:y1, x0:x1]


def apply_filter(img, mode='median3'):
    if mode == 'median3':
        return cv2.medianBlur(img, 3)           # removes sensor speckle, preserves chroma edges
    if mode == 'bilateral':
        return cv2.bilateralFilter(img, d=5, sigmaColor=35, sigmaSpace=35)
    return img


def load_preprocess_rgb(path, size, cfg):
    x = tiff.imread(path)
    if x.ndim == 4: x = x[0]
    if x.ndim == 3 and x.shape[0] in (1, 3, 4): x = np.transpose(x, (1, 2, 0))
    if x.ndim == 2: x = np.stack([x] * 3, axis=-1)
    if x.shape[-1] == 1: x = np.concatenate([x] * 3, axis=-1)
    if x.shape[-1] == 4: x = x[..., :3]

    x = stretch_to_uint8(x, cfg.get('stretch_mode', 'minmax'))

    crop_mode = cfg.get('crop_mode', 'none')
    if crop_mode == 'center':
        x = center_square_crop(x)
    elif crop_mode == 'content':
        x = content_crop(x, cfg.get('crop_margin', 0.04))

    x = apply_filter(x, cfg.get('filter_mode', 'none'))
    return cv2.resize(x, (size[1], size[0]), interpolation=cv2.INTER_AREA)


def convert_encoding(img_rgb_uint8, encoding):
    if encoding == 'RGB':
        return img_rgb_uint8.copy()
    return cv2.cvtColor(img_rgb_uint8, _CV2_FORWARD[encoding])


def normalize_channels_01(img_uint8, encoding):
    img = img_uint8.astype(np.float32)
    if encoding == 'HSV':
        out = img.copy()
        out[..., 0] /= 179.0     # OpenCV packs Hue into 0-179, not 0-255
        out[..., 1] /= 255.0
        out[..., 2] /= 255.0
        return out
    return img / 255.0

print('Preprocessing helpers ready (crop: none/center/content, filter: none/median3/bilateral)')

In [ ]:
def list_score_files(root, score_folders):
    """Resolve the on-disk files once so every preprocessing variant loads the
    exact same images in the exact same order."""
    if not os.path.exists(root):
        raise FileNotFoundError(f'NOCC_ROOT not found: {root}')

    files, labels = [], []
    for s in score_folders:
        candidates = [
            os.path.join(root, str(s)),
            os.path.join(root, f'Score={s:02d}'),
            os.path.join(root, f'Score={s}'),
            os.path.join(root, f'{s:02d}'),
        ]
        folder = next((p for p in candidates if os.path.isdir(p)), None)
        if folder is None:
            print(f'[Score {s}] folder missing')
            continue

        all_tifs = sorted(set(
            glob.glob(os.path.join(folder, '*.tif')) +
            glob.glob(os.path.join(folder, '*.TIF')) +
            glob.glob(os.path.join(folder, '*.tiff'))
        ))
        nocc = [p for p in all_tifs if 'nocc' in os.path.basename(p).lower()]
        chosen = nocc if nocc else all_tifs
        print(f'  [Score {s}] {len(chosen)} images')
        files.extend(chosen)
        labels.extend([s - 1] * len(chosen))

    if not files:
        raise RuntimeError('No images found under NOCC_ROOT.')
    return files, np.array(labels, dtype=np.int32)


def load_variant(files, cfg, size=IMG_SIZE):
    imgs = []
    for p in files:
        imgs.append(load_preprocess_rgb(p, size, cfg))
    return np.array(imgs, dtype=np.uint8)


TIF_FILES, y_raw = list_score_files(NOCC_ROOT, SCORES)
print(f'\nResolved {len(TIF_FILES)} images')

X_rgb = load_variant(TIF_FILES, PREPROCESS)
print(f'Main variant     : {X_rgb.shape}  ({PREPROCESS["crop_mode"]} crop + {PREPROCESS["filter_mode"]} filter)')

unique, counts = np.unique(y_raw + 1, return_counts=True)
print('Class counts     :', dict(zip(unique.tolist(), counts.tolist())))

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))
for ax, i in zip(axes, np.linspace(0, len(X_rgb) - 1, 4).astype(int)):
    ax.imshow(X_rgb[i]); ax.set_title(f'score {y_raw[i]+1}'); ax.axis('off')
fig.suptitle('Preprocessed samples (content crop + colour-safe filter)')
plt.tight_layout(); plt.show()

In [ ]:
# CRITICAL FIX vs the old draft:
# geometric augs run on RGB and are converted to the target encoding afterwards.
# Rotating/zooming an HSV array interpolates the circular Hue channel across the
# 179->0 wrap and fabricates hues, which is what crippled the HSV pipeline before.
# Only photometric augs (blur/noise) are safe to apply in the encoded domain.

def aug_flip(img, v):
    return cv2.flip(img, [1, 0, -1][v])

def aug_rotate(img, v):
    angles = [15, -15, 30]
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angles[v], 1.0)
    return cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)

def aug_zoom(img, v):
    f = [0.10, 0.20, 0.30][v]
    h, w = img.shape[:2]
    dh, dw = int(h * f / 2), int(w * f / 2)
    return cv2.resize(img[dh:h - dh, dw:w - dw], (w, h), interpolation=cv2.INTER_LINEAR)

def aug_blur(img, v):
    return cv2.GaussianBlur(img, (3, 3), [0.5, 1.0, 1.5][v])

def aug_noise(img, v):
    std = [3, 6, 9][v]
    n = np.random.normal(0, std, img.shape).astype(np.float32)
    return np.clip(img.astype(np.float32) + n, 0, 255).astype(np.uint8)

# (name, fn, domain, n_variants) -- 'rgb' = augment then encode, 'enc' = augment encoded
AUG_CATALOGUE = [
    ('Flip',   aug_flip,   'rgb', 3),
    ('Rotate', aug_rotate, 'rgb', 3),
    ('Zoom',   aug_zoom,   'rgb', 3),
    ('Blur',   aug_blur,   'enc', 3),
    ('Noise',  aug_noise,  'enc', 3),
]
AUG_FACTOR = 1 + sum(n for *_, n in AUG_CATALOGUE)
print(f'Offline augmentation factor: x{AUG_FACTOR}  (matches the originals)')

In [ ]:
class EncodedDataset(Dataset):
    def __init__(self, enc_imgs, labels, encoding, augment_online=False):
        normed = np.stack([normalize_channels_01(x, encoding) for x in enc_imgs])
        self.imgs = torch.from_numpy(normed.astype(np.float32)).permute(0, 3, 1, 2)
        self.labels = labels.astype(np.int64)
        self.scores = torch.from_numpy(IDX_TO_SCORE[self.labels]).unsqueeze(1)
        self.augment_online = augment_online

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.imgs[idx].clone()
        if self.augment_online and random.random() < 0.3:
            if random.random() < 0.5: img = img.flip(2)
            if random.random() < 0.5: img = img.flip(1)
        return img, self.scores[idx], self.labels[idx]


# One split set, reused by every pipeline and every ablation arm -> differences are the method, not the data
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
fold_splits = list(skf.split(np.zeros(len(y_raw)), y_raw))
for i, (tr, va) in enumerate(fold_splits):
    print(f'  Fold {i+1}: train={len(tr)}  val={len(va)}')

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, drop=0.2, pool=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        ]
        if pool:
            layers.append(nn.MaxPool2d(2))
        layers.append(nn.Dropout2d(drop))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class CNNBackbone4(nn.Module):
    def __init__(self, in_ch=3):
        super().__init__()
        self.blocks = nn.Sequential(
            ConvBlock(in_ch, 32, 0.10),
            ConvBlock(32, 64, 0.15),
            ConvBlock(64, 128, 0.20),
            ConvBlock(128, 256, 0.25),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.out_dim = 256

    def forward(self, x):
        return self.gap(self.blocks(x)).flatten(1)


class CNNBackbone6(nn.Module):
    def __init__(self, in_ch=3):
        super().__init__()
        self.blocks = nn.Sequential(
            ConvBlock(in_ch, 32, 0.10),
            ConvBlock(32, 64, 0.15),
            ConvBlock(64, 128, 0.20),
            ConvBlock(128, 192, 0.25),
            ConvBlock(192, 256, 0.30),
            ConvBlock(256, 384, 0.30, pool=False),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.out_dim = 384

    def forward(self, x):
        return self.gap(self.blocks(x)).flatten(1)


class DualHeadNet(nn.Module):
    """Shared trunk -> ordinal head (NUM_ORD logits) + regression scalar."""
    def __init__(self, backbone='cnn4'):
        super().__init__()
        self.backbone = CNNBackbone6() if backbone == 'cnn6' else CNNBackbone4()
        self.shared = nn.Sequential(
            nn.Linear(self.backbone.out_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.35)
        )
        self.ord_head = nn.Linear(128, NUM_ORD)
        self.reg_head = nn.Linear(128, 1)

    def forward(self, x):
        z = self.shared(self.backbone(x))
        return self.ord_head(z), self.reg_head(z)


class RegressionCNN(nn.Module):
    """Benchmark baseline ported from 260305_cnn_regression_analysis:
    6 double-conv stages 16->512 with a single regression output. GAP replaces the
    original flatten so it runs at 128x128 with the same channel schedule."""
    def __init__(self, in_ch=3, depth=16):
        super().__init__()
        d = depth
        self.features = nn.Sequential(
            ConvBlock(in_ch,  d,      0.05),
            ConvBlock(d,      d * 2,  0.10),
            ConvBlock(d * 2,  d * 4,  0.15),
            ConvBlock(d * 4,  d * 8,  0.20),
            ConvBlock(d * 8,  d * 16, 0.25),
            ConvBlock(d * 16, d * 32, 0.25, pool=False),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Linear(d * 32, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.35),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.20),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        z = self.gap(self.features(x)).flatten(1)
        return None, self.head(z)      # (ord_logits, reg_out) -> ordinal branch unused


def build_model(cfg):
    if cfg['model'] == 'regression_cnn':
        return RegressionCNN(in_ch=3, depth=16)
    return DualHeadNet(backbone=cfg['backbone'])


for _name, _m in [('DualHeadNet(cnn6)', DualHeadNet('cnn6')),
                  ('DualHeadNet(cnn4)', DualHeadNet('cnn4')),
                  ('RegressionCNN', RegressionCNN())]:
    print(f'{_name:22s}: {sum(p.numel() for p in _m.parameters() if p.requires_grad):,} params')

In [ ]:
def ordinal_targets(y_score):
    thresholds = torch.arange(1, NUM_ORD + 1, dtype=torch.float32, device=y_score.device).unsqueeze(0)
    return (y_score.unsqueeze(1) > thresholds).float()

def loss_ordinal_coral(y_score, ord_logits):
    return F.binary_cross_entropy_with_logits(ord_logits, ordinal_targets(y_score))

def loss_ordinal_corn(y_score, ord_logits):
    y_ord = ordinal_targets(y_score)
    mask = torch.ones_like(y_ord)
    mask[:, 1:] = y_ord[:, :-1]
    bce = F.binary_cross_entropy_with_logits(ord_logits, y_ord, reduction='none')
    return (bce * mask).sum() / mask.sum().clamp(min=1.0)

def loss_ranking(reg_out, target, margin=1.0):
    pred = reg_out.squeeze(1); t = target.squeeze(1)
    gap = t.unsqueeze(1) - t.unsqueeze(0)
    mask = (gap.abs() >= 1.0).float()
    l = torch.clamp(margin - gap.sign() * (pred.unsqueeze(1) - pred.unsqueeze(0)), min=0.0)
    return (mask * l).sum() / (mask.sum() + 1e-8)

def composite_loss(ord_logits, reg_out, target, ordinal_mode='coral'):
    """1.0*ordinal + 0.5*huber + 0.5*rank -- the weights validated in the originals."""
    y_score = target.squeeze(1)
    l_ord = loss_ordinal_corn(y_score, ord_logits) if ordinal_mode == 'corn' else loss_ordinal_coral(y_score, ord_logits)
    return 1.0 * l_ord + 0.5 * F.huber_loss(reg_out, target, delta=1.0) + 0.5 * loss_ranking(reg_out, target)

def regression_loss(ord_logits, reg_out, target, ordinal_mode=None):
    """Single-head arm: same robust regression + ranking terms, no ordinal branch."""
    return F.huber_loss(reg_out, target, delta=1.0) + 0.5 * loss_ranking(reg_out, target)

LOSS_FNS = {'composite': composite_loss, 'regression': regression_loss}


def decode_ordinal(ord_logits, mode='coral'):
    p = torch.sigmoid(ord_logits)
    if mode == 'coral':
        return 1.0 + p.sum(dim=1, keepdim=True)
    return 1.0 + (torch.cumprod(p, dim=1) > 0.5).float().sum(dim=1, keepdim=True)


def compute_metrics(preds, targets):
    preds = np.asarray(preds, dtype=np.float32).ravel()
    targets = np.asarray(targets, dtype=np.float32).ravel()
    p_int = np.clip(np.round(preds), 1, NUM_CLASSES).astype(int)
    t_int = np.clip(np.round(targets), 1, NUM_CLASSES).astype(int)
    return {
        'r2': float(r2_score(targets, preds)),
        'rmse': float(np.sqrt(mean_squared_error(targets, preds))),
        'mae': float(mean_absolute_error(targets, preds)),
        'rho': float(spearmanr(targets, preds)[0]),
        'tau': float(kendalltau(targets, preds)[0]),
        'qwk': float(cohen_kappa_score(t_int, p_int, weights='quadratic')),
        'acc0': float(np.mean(np.abs(p_int - t_int) <= 0)),
        'acc1': float(np.mean(np.abs(p_int - t_int) <= 1)),
    }

print('Losses: composite (CORAL/CORN) | regression (huber+rank)   Metrics: r2 rmse mae rho tau qwk acc0 acc1')

In [ ]:
def build_encoded(rgb_imgs, encoding):
    if encoding == 'RGB':
        return rgb_imgs.copy()
    return np.array([convert_encoding(x, encoding) for x in rgb_imgs], dtype=np.uint8)


def augment_fold(rgb_all, enc_all, train_idx, encoding):
    """Geometric augs are generated from RGB and encoded afterwards; photometric
    augs act on the encoded image. Only the training split is expanded."""
    xs, ys = [], []
    for i in train_idx:
        xs.append(enc_all[i]); ys.append(y_raw[i])

    for _, fn, domain, n_var in AUG_CATALOGUE:
        for v in range(n_var):
            for i in train_idx:
                if domain == 'rgb':
                    xs.append(convert_encoding(fn(rgb_all[i], v), encoding))
                else:
                    xs.append(fn(enc_all[i], v))
                ys.append(y_raw[i])

    return np.array(xs, dtype=np.uint8), np.array(ys, dtype=np.int32)


def _predict_loader(model, loader, cfg):
    reg_l, ord_l, blend_l = [], [], []
    model.eval()
    with torch.no_grad():
        for img, _, _ in loader:
            img = img.to(device)
            ord_logits, reg_out = model(img)
            reg = reg_out.clamp(1.0, 9.0)
            if cfg['head'] == 'dual':
                ordv = decode_ordinal(ord_logits, mode=cfg['ordinal_mode']).clamp(1.0, 9.0)
                blend = BLEND_W_ORD * ordv + (1.0 - BLEND_W_ORD) * reg
            else:
                ordv = reg
                blend = reg
            reg_l.append(reg.cpu().numpy().ravel())
            ord_l.append(ordv.cpu().numpy().ravel())
            blend_l.append(blend.cpu().numpy().ravel())
    return np.concatenate(reg_l), np.concatenate(ord_l), np.concatenate(blend_l)


def train_one_fold(cfg, rgb_all, enc_all, fold_idx, tr_idx, va_idx,
                   epochs, es_patience, save_ckpt=True, tag=''):
    x_tr, y_tr = augment_fold(rgb_all, enc_all, tr_idx, cfg['encoding'])
    x_va, y_va = enc_all[va_idx], y_raw[va_idx]

    dl_tr = DataLoader(EncodedDataset(x_tr, y_tr, cfg['encoding'], augment_online=True),
                       batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
                       drop_last=(len(x_tr) % BATCH_SIZE == 1))
    dl_va = DataLoader(EncodedDataset(x_va, y_va, cfg['encoding'], augment_online=False),
                       batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = build_model(cfg).to(device)
    loss_fn = LOSS_FNS[cfg['loss']]
    opt = torch.optim.Adam(model.parameters(), lr=LR_INIT)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='max', factor=LR_FACTOR, patience=PATIENCE_LR, min_lr=LR_MIN)

    true_va = IDX_TO_SCORE[y_va]
    best_r2, best_state, best_epoch, no_improve = -np.inf, None, 0, 0
    history = {'train_loss': [], 'val_loss': [], 'val_r2': [], 'lr': []}

    for ep in range(epochs):
        model.train()
        tr_losses = []
        for img, score, _ in dl_tr:
            img, score = img.to(device), score.to(device)
            opt.zero_grad()
            ord_logits, reg_out = model(img)
            loss = loss_fn(ord_logits, reg_out, score, cfg.get('ordinal_mode'))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
            tr_losses.append(float(loss.item()))

        model.eval()
        va_losses = []
        with torch.no_grad():
            for img, score, _ in dl_va:
                img, score = img.to(device), score.to(device)
                ord_logits, reg_out = model(img)
                va_losses.append(float(loss_fn(ord_logits, reg_out, score, cfg.get('ordinal_mode')).item()))

        _, _, blend_va = _predict_loader(model, dl_va, cfg)
        val_r2 = float(r2_score(true_va, blend_va))

        history['train_loss'].append(float(np.mean(tr_losses)))
        history['val_loss'].append(float(np.mean(va_losses)))
        history['val_r2'].append(val_r2)
        history['lr'].append(opt.param_groups[0]['lr'])
        sch.step(val_r2)

        if val_r2 > best_r2 + 1e-5:
            best_r2, best_epoch, no_improve = val_r2, ep, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            if no_improve >= es_patience:
                break

    model.load_state_dict(best_state)
    ckpt = None
    if save_ckpt:
        ckpt = os.path.join(OUT_DIR, 'checkpoints', f"{tag}{cfg['name']}_fold{fold_idx+1}.pt")
        torch.save(model.state_dict(), ckpt)

    reg, ordv, blend = _predict_loader(model, dl_va, cfg)
    return dict(reg=reg, ord=ordv, blend=blend, history=history,
                best_epoch=best_epoch + 1, n_epochs=len(history['val_r2']), ckpt=ckpt)


def run_pipeline(name, cfg, rgb_all=None, epochs=None, es_patience=None,
                 save_ckpt=True, tag='', verbose=True):
    cfg = dict(cfg); cfg['name'] = name
    cfg.setdefault('head', 'dual')
    cfg.setdefault('loss', 'composite')
    cfg.setdefault('ordinal_mode', 'coral')

    rgb_all = X_rgb if rgb_all is None else rgb_all
    epochs = EPOCHS if epochs is None else epochs
    es_patience = PATIENCE_ES if es_patience is None else es_patience
    enc_all = build_encoded(rgb_all, cfg['encoding'])

    n = len(y_raw)
    oof = {k: np.full(n, np.nan, dtype=np.float32) for k in ('reg', 'ord', 'blend')}
    fold_rows, histories = [], []

    t0 = time.time()
    for fi, (tr_idx, va_idx) in enumerate(fold_splits):
        out = train_one_fold(cfg, rgb_all, enc_all, fi, tr_idx, va_idx,
                             epochs, es_patience, save_ckpt=save_ckpt, tag=tag)
        for k in ('reg', 'ord', 'blend'):
            oof[k][va_idx] = out[k]
        histories.append(out['history'])

        m = compute_metrics(out['blend'], IDX_TO_SCORE[y_raw[va_idx]])
        m['fold'] = fi + 1
        m['best_epoch'] = out['best_epoch']
        m['epochs_run'] = out['n_epochs']
        fold_rows.append(m)
        if verbose:
            print(f"    [{name}] fold {fi+1}/{K_FOLDS}  best_ep={out['best_epoch']}/{out['n_epochs']}  R2={m['r2']:.4f}")

    true_all = IDX_TO_SCORE[y_raw]
    oof_metrics = {src: compute_metrics(oof[src], true_all) for src in ('reg', 'ord', 'blend')}
    minutes = (time.time() - t0) / 60.0
    if verbose:
        print(f"  {name} done in {minutes:.1f} min | blend OOF R2={oof_metrics['blend']['r2']:.4f} "
              f"QWK={oof_metrics['blend']['qwk']:.4f}")

    return dict(cfg=cfg, oof_reg=oof['reg'], oof_ord=oof['ord'], oof_blend=oof['blend'],
                true=true_all, fold_metrics=pd.DataFrame(fold_rows), histories=histories,
                oof_metrics=oof_metrics, minutes=minutes)

print('Engine ready: run_pipeline(name, cfg, rgb_all=...) -> OOF reg/ord/blend + per-fold metrics')

In [ ]:
# === 2 · Why the old draft lost R2 -- measured, not asserted ==================
# Every arm uses the SAME winner config (cnn6 + HSV + CORAL) and the SAME folds.
# Only preprocessing / augmentation domain changes. Reduced budget: relative
# ranking is what matters here, not absolute values.
ABL_EPOCHS, ABL_ES = (20, 6) if QUICK_MODE else (80, 15)

PROBE_CFG = dict(model='dual', backbone='cnn6', encoding='HSV',
                 head='dual', loss='composite', ordinal_mode='coral')

ABLATION_ARMS = {
    'originals (no crop, no filter, minmax)': dict(prep=LEGACY_PREPROCESS, legacy_aug=False),
    'content crop + median3 (this notebook)': dict(prep=PREPROCESS,        legacy_aug=False),
    'old draft (center crop + bilateral + pct)': dict(
        prep={'crop_mode': 'center', 'filter_mode': 'bilateral', 'stretch_mode': 'percentile', 'crop_margin': 0.0},
        legacy_aug=False),
    'old draft aug bug (augment in HSV space)': dict(prep=PREPROCESS, legacy_aug=True),
}

_variant_cache = {}
def get_variant(prep_cfg):
    key = tuple(sorted(prep_cfg.items()))
    if key not in _variant_cache:
        _variant_cache[key] = load_variant(TIF_FILES, prep_cfg)
    return _variant_cache[key]

_AUG_BACKUP = list(AUG_CATALOGUE)
ablation_rows = []

for arm_name, arm in ABLATION_ARMS.items():
    print(f'\n--- ablation arm: {arm_name} ---')
    rgb_variant = get_variant(arm['prep'])

    if arm['legacy_aug']:
        # reproduce the bug: every augmentation applied to the encoded image
        AUG_CATALOGUE[:] = [(n, f, 'enc', k) for (n, f, _d, k) in _AUG_BACKUP]
    else:
        AUG_CATALOGUE[:] = _AUG_BACKUP

    res = run_pipeline('abl', PROBE_CFG, rgb_all=rgb_variant,
                       epochs=ABL_EPOCHS, es_patience=ABL_ES,
                       save_ckpt=False, tag='abl_', verbose=False)
    m = res['oof_metrics']['blend']
    ablation_rows.append({'arm': arm_name, 'R2': m['r2'], 'RMSE': m['rmse'],
                          'MAE': m['mae'], 'QWK': m['qwk'], 'Acc@1': m['acc1']})
    print(f"    blend OOF R2={m['r2']:.4f}  QWK={m['qwk']:.4f}")

AUG_CATALOGUE[:] = _AUG_BACKUP     # always restore the correct catalogue

df_abl = pd.DataFrame(ablation_rows).sort_values('R2', ascending=False).reset_index(drop=True)
df_abl.to_csv(os.path.join(OUT_DIR, 'preprocessing_ablation.csv'), index=False)
display(df_abl)

fig, ax = plt.subplots(figsize=(11, 4))
ax.barh(df_abl['arm'][::-1], df_abl['R2'][::-1], color='steelblue', edgecolor='black')
ax.set_xlabel('blend OOF R2'); ax.set_title('Where the R2 gap came from')
for i, v in enumerate(df_abl['R2'][::-1]):
    ax.text(v, i, f' {v:.4f}', va='center')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'preprocessing_ablation.png'), dpi=150); plt.show()

In [ ]:
# === 3 · The benchmark: 3 pipelines, identical protocol ======================
PIPELINES = {
    'A_cnn6_HSV_coral': dict(
        label='A: 6-block CNN + HSV + CORAL dual-head',
        model='dual', backbone='cnn6', encoding='HSV',
        head='dual', loss='composite', ordinal_mode='coral'),

    'B_cnn4_LAB_coral': dict(
        label='B: 4-block CNN + LAB + CORAL dual-head',
        model='dual', backbone='cnn4', encoding='LAB',
        head='dual', loss='composite', ordinal_mode='coral'),

    'C_regression_cnn_RGB': dict(
        label='C: RegressionCNN (16->512) + RGB + Huber/rank',
        model='regression_cnn', backbone='regression_cnn', encoding='RGB',
        head='reg', loss='regression', ordinal_mode=None),
}

print('=' * 84)
print(f'BENCHMARK | {K_FOLDS}-fold CV | {EPOCHS} epochs | ES={PATIENCE_ES} | LR patience={PATIENCE_LR}')
print(f'          | crop={PREPROCESS["crop_mode"]} filter={PREPROCESS["filter_mode"]} aug=x{AUG_FACTOR}')
print('=' * 84)

RESULTS = {}
for name, cfg in PIPELINES.items():
    print(f'\n### {cfg["label"]}')
    RESULTS[name] = run_pipeline(name, cfg, rgb_all=X_rgb, tag='final_')

print('\nAll pipelines trained.')

In [ ]:
METRIC_COLS = ['r2', 'rmse', 'mae', 'rho', 'tau', 'qwk', 'acc0', 'acc1']

# Headline table: best scoring source per pipeline, OOF + cross-fold spread
summary_rows = []
for name, res in RESULTS.items():
    m = res['oof_metrics']['blend']
    fm = res['fold_metrics']
    summary_rows.append({
        'pipeline': name,
        'label': res['cfg']['label'],
        'encoding': res['cfg']['encoding'],
        'head': res['cfg']['head'],
        'R2': m['r2'],
        'R2_cv_mean': fm['r2'].mean(),
        'R2_cv_std': fm['r2'].std(),
        'RMSE': m['rmse'], 'MAE': m['mae'], 'QWK': m['qwk'],
        'Acc@0': m['acc0'], 'Acc@1': m['acc1'],
        'Spearman': m['rho'], 'Kendall': m['tau'],
        'mean_best_epoch': fm['best_epoch'].mean(),
        'minutes': res['minutes'],
    })

df_summary = pd.DataFrame(summary_rows).sort_values('R2', ascending=False).reset_index(drop=True)
df_summary.to_csv(os.path.join(OUT_DIR, 'benchmark_summary.csv'), index=False)
print('HEADLINE COMPARISON (blend / OOF)')
display(df_summary.round(4))

# Per-scoring-source breakdown: reg vs ord vs blend for each pipeline
src_rows = []
for name, res in RESULTS.items():
    for src in ('reg', 'ord', 'blend'):
        m = res['oof_metrics'][src]
        row = {'pipeline': name, 'source': src}
        row.update({k: m[k] for k in METRIC_COLS})
        src_rows.append(row)

df_sources = pd.DataFrame(src_rows)
df_sources.to_csv(os.path.join(OUT_DIR, 'benchmark_by_source.csv'), index=False)
print('\nPER-SOURCE BREAKDOWN (reg / ord / blend)')
display(df_sources.round(4))

for name, res in RESULTS.items():
    res['fold_metrics'].to_csv(os.path.join(OUT_DIR, f'{name}_fold_metrics.csv'), index=False)
print('\nSaved CSVs ->', OUT_DIR)

In [ ]:
from itertools import combinations

# --- Pairwise significance on the shared folds + bootstrap CI on OOF R2 delta ---
rng = np.random.default_rng(SEED)
n_obs = len(y_raw)

stat_rows = []
for a, b in combinations(RESULTS.keys(), 2):
    fa = RESULTS[a]['fold_metrics']['r2'].values
    fb = RESULTS[b]['fold_metrics']['r2'].values
    t_stat, p_val = ttest_rel(fa, fb)

    ea = (RESULTS[a]['oof_blend'] - RESULTS[a]['true']) ** 2
    eb = (RESULTS[b]['oof_blend'] - RESULTS[b]['true']) ** 2
    boot = []
    for _ in range(4000):
        idx = rng.integers(0, n_obs, n_obs)
        boot.append(eb[idx].mean() - ea[idx].mean())     # >0 means A has lower error
    lo, hi = np.percentile(boot, [2.5, 97.5])

    stat_rows.append({
        'A': a, 'B': b,
        'foldR2_mean_delta': fa.mean() - fb.mean(),
        't_stat': t_stat, 'p_value': p_val,
        'MSE_gain_A_over_B': (eb.mean() - ea.mean()),
        'boot_CI_low': lo, 'boot_CI_high': hi,
        'significant_95': bool(lo > 0 or hi < 0),
    })

df_stats = pd.DataFrame(stat_rows)
df_stats.to_csv(os.path.join(OUT_DIR, 'pairwise_significance.csv'), index=False)
print('PAIRWISE COMPARISON (paired t-test on folds + bootstrap CI on squared error)')
display(df_stats.round(4))

# --- Per-fold R2 stability ---
fold_tbl = pd.DataFrame({name: res['fold_metrics']['r2'].values for name, res in RESULTS.items()})
fold_tbl.index = [f'fold{i+1}' for i in range(len(fold_tbl))]
display(fold_tbl.round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
fold_tbl.plot(kind='bar', ax=axes[0]); axes[0].set_title('Per-fold R2'); axes[0].set_ylabel('R2'); axes[0].grid(alpha=0.3)
axes[0].legend(fontsize=8)
sns.boxplot(data=fold_tbl, ax=axes[1]); axes[1].set_title('R2 spread across folds'); axes[1].grid(alpha=0.3)
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fold_stability.png'), dpi=150); plt.show()

# --- Training curves ---
fig, axes = plt.subplots(1, len(RESULTS), figsize=(6 * len(RESULTS), 4), squeeze=False)
for ax, (name, res) in zip(axes[0], RESULTS.items()):
    for fi, h in enumerate(res['histories']):
        ax.plot(h['val_r2'], alpha=0.8, label=f'fold {fi+1}')
    ax.set_title(f'{name}\nval R2'); ax.set_xlabel('epoch'); ax.grid(alpha=0.3); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'training_curves.png'), dpi=150); plt.show()

In [ ]:
def plot_confusion_rownorm(preds, true, title, save_name):
    labels = list(range(1, NUM_CLASSES + 1))
    p_int = np.clip(np.round(preds), 1, NUM_CLASSES).astype(int)
    t_int = np.clip(np.round(true), 1, NUM_CLASSES).astype(int)
    cm = confusion_matrix(t_int, p_int, labels=labels)
    cmn = cm.astype(np.float64) / cm.sum(axis=1, keepdims=True).clip(min=1)

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(labels)
    ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(labels)
    ax.set_xlabel('Predicted score'); ax.set_ylabel('True score'); ax.set_title(title)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            if cm[i, j] > 0:
                ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                        color='white' if cmn[i, j] > 0.5 else 'black', fontsize=9)
    fig.colorbar(im, ax=ax, label='Row-normalized recall')
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, save_name), dpi=150); plt.show()
    return cm

for name, res in RESULTS.items():
    plot_confusion_rownorm(res['oof_blend'], res['true'],
                           f'{name} — OOF confusion (R2={res["oof_metrics"]["blend"]["r2"]:.4f})',
                           f'{name}_confusion.png')

# Per-class error, so a winner is not chosen on a global average that hides a weak class
perclass = []
for name, res in RESULTS.items():
    for c in range(1, NUM_CLASSES + 1):
        idx = np.where(np.round(res['true']) == c)[0]
        if len(idx) == 0:
            continue
        pred_c = res['oof_blend'][idx]
        perclass.append({'pipeline': name, 'score': c, 'n': len(idx),
                         'mean_pred': pred_c.mean(),
                         'MAE': np.abs(pred_c - res['true'][idx]).mean(),
                         'Acc@1': np.mean(np.abs(np.round(pred_c) - c) <= 1)})

df_perclass = pd.DataFrame(perclass)
df_perclass.to_csv(os.path.join(OUT_DIR, 'per_class_stats.csv'), index=False)

fig, ax = plt.subplots(figsize=(11, 4))
sns.barplot(data=df_perclass, x='score', y='MAE', hue='pipeline', ax=ax)
ax.set_title('Per-class MAE (lower is better)'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'per_class_mae.png'), dpi=150); plt.show()
display(df_perclass.round(3))

In [ ]:
def plot_frieze_by_class(images_rgb, values, true_scores, title, max_per_class=10, save_name=None):
    """One row per true class, images sorted left->right by predicted score.
    A good model shows a smooth colour ramp along each row and rising row means."""
    labels = list(range(1, NUM_CLASSES + 1))
    true_int = np.clip(np.round(true_scores), 1, NUM_CLASSES).astype(int)
    img01 = images_rgb.astype(np.float32) / 255.0

    fig, axes = plt.subplots(len(labels), max_per_class,
                             figsize=(max_per_class * 1.9, len(labels) * 1.7), squeeze=False)
    for r, c in enumerate(labels):
        idx = np.where(true_int == c)[0]
        if len(idx):
            idx = idx[np.argsort(values[idx])]
        if len(idx) > max_per_class:
            idx = idx[np.linspace(0, len(idx) - 1, max_per_class).round().astype(int)]

        for col in range(max_per_class):
            ax = axes[r][col]
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_visible(False)
            if col < len(idx):
                j = idx[col]
                ax.imshow(np.clip(img01[j], 0, 1))
                err = abs(values[j] - true_scores[j])
                ax.set_title(f'{values[j]:.2f}', fontsize=8,
                             color='darkred' if err > 1.0 else 'black')
            if col == 0:
                ax.set_ylabel(f'S{c}', fontsize=10, fontweight='bold')

    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.005)
    plt.tight_layout()
    if save_name:
        plt.savefig(os.path.join(OUT_DIR, save_name), dpi=140, bbox_inches='tight')
    plt.show()

for name, res in RESULTS.items():
    plot_frieze_by_class(X_rgb, res['oof_blend'], res['true'],
                         f'{name} — OOF prediction frieze (red title = |error| > 1)',
                         save_name=f'{name}_frieze.png')

In [ ]:
# === 6 · Decision: which pipeline to ship ====================================
# Composite score keeps the choice from hanging on a single metric: accuracy
# (R2, QWK, Acc@1) is rewarded, cross-fold instability is penalised.
rank = df_summary.copy()
rank['R2_norm']   = (rank['R2']   - rank['R2'].min())   / (rank['R2'].max()   - rank['R2'].min()   + 1e-9)
rank['QWK_norm']  = (rank['QWK']  - rank['QWK'].min())  / (rank['QWK'].max()  - rank['QWK'].min()  + 1e-9)
rank['Acc1_norm'] = (rank['Acc@1']- rank['Acc@1'].min())/ (rank['Acc@1'].max()- rank['Acc@1'].min()+ 1e-9)
rank['stability'] = 1.0 - (rank['R2_cv_std'] - rank['R2_cv_std'].min()) / (rank['R2_cv_std'].max() - rank['R2_cv_std'].min() + 1e-9)
rank['decision_score'] = (0.40 * rank['R2_norm'] + 0.25 * rank['QWK_norm'] +
                          0.20 * rank['Acc1_norm'] + 0.15 * rank['stability'])
rank = rank.sort_values('decision_score', ascending=False).reset_index(drop=True)
rank.to_csv(os.path.join(OUT_DIR, 'decision_ranking.csv'), index=False)

display(rank[['pipeline', 'label', 'R2', 'QWK', 'Acc@1', 'R2_cv_std', 'decision_score']].round(4))

win = rank.iloc[0]
runner = rank.iloc[1] if len(rank) > 1 else None

print('=' * 88)
print('FINAL RECOMMENDATION')
print('=' * 88)
print(f"Winner        : {win['pipeline']}")
print(f"Configuration : {win['label']}")
print(f"OOF metrics   : R2={win['R2']:.4f}  RMSE={win['RMSE']:.4f}  MAE={win['MAE']:.4f}  "
      f"QWK={win['QWK']:.4f}  Acc@1={win['Acc@1']*100:.1f}%")
print(f"Stability     : R2 = {win['R2_cv_mean']:.4f} +- {win['R2_cv_std']:.4f} across {K_FOLDS} folds")
print(f"Protocol      : crop={PREPROCESS['crop_mode']}, filter={PREPROCESS['filter_mode']}, "
      f"aug=x{AUG_FACTOR}, {EPOCHS}ep, ES={PATIENCE_ES}, LRpat={PATIENCE_LR}")

if runner is not None:
    pair = df_stats[((df_stats['A'] == win['pipeline']) & (df_stats['B'] == runner['pipeline'])) |
                    ((df_stats['B'] == win['pipeline']) & (df_stats['A'] == runner['pipeline']))]
    if len(pair):
        r = pair.iloc[0]
        verdict = 'STATISTICALLY SIGNIFICANT' if r['significant_95'] else 'NOT significant (overlapping 95% CI)'
        print(f"\nvs runner-up  : {runner['pipeline']}  (dR2 = {win['R2'] - runner['R2']:+.4f})")
        print(f"Significance  : {verdict}  [p={r['p_value']:.4f}, bootCI=({r['boot_CI_low']:.4f}, {r['boot_CI_high']:.4f})]")
        if not r['significant_95']:
            print('Guidance      : the top two are statistically tied -> prefer the cheaper/simpler model.')
print('=' * 88)

# Dashboard
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, ['R2', 'QWK', 'Acc@1', 'decision_score']):
    sns.barplot(data=rank, x='pipeline', y=col, ax=ax)
    ax.set_title(col); ax.tick_params(axis='x', rotation=20); ax.grid(alpha=0.25)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'final_dashboard.png'), dpi=150); plt.show()

for name, res in RESULTS.items():
    np.save(os.path.join(OUT_DIR, f'{name}_oof_blend.npy'), res['oof_blend'])
np.save(os.path.join(OUT_DIR, 'oof_true.npy'), RESULTS[list(RESULTS)[0]]['true'])
print('Artifacts saved ->', OUT_DIR)